# EDA Notebook
**Education Under Fire** — exploratory scratch space.

This notebook is for exploration only. Production cleaning code lives in the numbered `.py` scripts.

Log issues in `docs/revision_notes.md`. The things most likely to need decisions are:
- **Event type reclassification (~2016):** "Battle-No change of territory" became several sub-types
- **Fatality recording:** early years have more zeros and unknowns
- **Admin2 (LGA) name inconsistencies:** spelling varies across years, which will matter for spatial merge
- **Actor naming:** Boko Haram appears under several names across years

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import geopandas as gpd
import sys

# Settings
sys.path.insert(0, '.')
from config import *

# Notebook display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('Setup complete')

## 1. ACLED — First look

In [ ]:
# Define paths and constants
ACLED_DIR = Path("../data/raw/acled")

# Load annual ACLED files into a dict
files = sorted(ACLED_DIR.glob("*.csv"))
annual = {}
for f in files:
    year = f.stem[-4:]
    annual[year] = pd.read_csv(f)
    print(f"{year}: {len(annual[year]):>6,} rows | {annual[year].shape[1]} columns")


## Column Overview

In [ ]:
# Check column names by year
col_sets = {yr: set(df.columns) for yr, df in annual.items()}

# Columns present in all years
always_present = set.intersection(*col_sets.values())
print(f"Columns present in all years ({len(always_present)}): ", sorted(always_present))

# Columns present in some years only
all_cols = set.union(*col_sets.values())
sometimes_present = all_cols - always_present

if sometimes_present:
    print(f"\nColumns present in some years only ({len(sometimes_present)}): ", sorted(sometimes_present))
    for col in sorted(sometimes_present):
        years_with = [yr for yr, s in col_sets.items() if col in s]
        print(f"  {col:35s} present in: {years_with}")
else:
    print("\nAll columns are present in all years.")

In [ ]:
# Dtypes by year
dtype_audit = {}
for yr, df in annual.items():
    for col in always_present:
        dtype_audit.setdefault(col, {})[yr] = str(df[col].dtype)

print("\nColumns with inconsistent dtypes:")
for col, yr_types in dtype_audit.items():
    unique_types = set(yr_types.values())
    if len(unique_types) > 1:
        print(f"  {col}: {yr_types}")

In [ ]:
# Missingness by year for key columns
key_cols = ["event_type", "fatalities", "admin1", "admin2", "latitude", "longitude"]

missing = {}
for yr, df in annual.items():
    missing[yr] = {
        col: df[col].isna().mean() for col in key_cols if col in df.columns
    }

missing_df = pd.DataFrame(missing).T
print(missing_df.to_string())

In [ ]:
# Unique values in categorical columns - to catch definition changes
cat_cols = ["event_type", "sub_event_type", "interaction"]

for col in cat_cols:
    print(f"\nUnique values in '{col}' by year:")
    for yr, df in annual.items():
        if col in df.columns:
            vals = sorted(df[col].dropna().unique())
            print(f"  {yr}: {len(vals)} unique values")
#            print(f"  {yr}: {vals}")

After harmonizing, concatenate ACLED data with a source year column before running content EDA

In [ ]:
# Conactenate with year label
df = pd.concat(
    [d.assign(source_year = yr) for yr, d in annual.items()],
    ignore_index=True
)

print(f"Combined: {len(df):,} rows")

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df.describe()

df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")


In [ ]:
# Events over time
events_per_year = df.groupby("source_year").size()
fatalities_per_year = df.groupby("source_year")["fatalities"].sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize = (10, 7), sharex = True)

events_per_year.plot(ax=ax1, marker='o', color = "#2166AC")
ax1.set_title("ACLED Events per Year - Nigeria")
ax1.set_ylabel("Number of Events")
ax1.axvline(x="2009", color="red", linestyle="--", alpha=0.5, label="Treatment year")

fatalities_per_year.plot(ax=ax2, marker='o', color = "#D6604D")
ax2.set_title("ACLED Fatalities per Year - Nigeria")
ax2.set_ylabel("Total Fatalities")
ax2.axvline(x="2009", color="red", linestyle="--", alpha=0.5)
plt.tight_layout()

# acled['year'].value_counts().sort_index().plot(kind='bar', title='ACLED Events per Year')
# plt.tight_layout()

In [ ]:
# Event types breakdown over time
event_type_year = (
    df.groupby(["source_year", "event_type"])
    .size()
    .unstack(fill_value=0)
)

event_type_year.plot(kind='bar', stacked=True, figsize=(12,5))
plt.title("Event Types by Year (stacked)")
plt.tight_layout()

In [ ]:
# Fatalities distribution

# Geographic coverage - states represented each year
states_per_year = df.groupby("source_year")["admin1"].nunique()
print("Unique states per year:")
print(states_per_year.to_string())

# Flag years where coverage looks thin
thin = states_per_year[states_per_year < states_per_year.median() * 0.8]
if len(thin):
#    print(f"\nPotentially thin coverage years (fewer than {states_per_year.median() * 0.8:.1f} states):")
    print(f"\nPotentially thin coverage: {thin.index.tolist()}")

# acled['fatalities'].describe()
# acled['fatalities'].hist(bins=50, log=True)
# plt.title('Fatalities per event (log scale)')

## 2. DHS — First look

In [ ]:
# import pyreadstat
# dhs, meta = pyreadstat.read_dta(DHS_FILE)
# print(dhs.shape)
# dhs.head()

In [ ]:
# Browse variable labels (Stata metadata)
# for var, label in meta.column_labels.items():
#     print(f'{var:30s} {label}')

In [ ]:
# School attendance by gender
# dhs.groupby('hv104')['hv121'].mean().rename({1: 'Male', 2: 'Female'}).plot(
#     kind='bar', title='School Attendance Rate by Sex'
# )

## 3. Geospatial — First look

In [ ]:
# Load and plot Nigeria LGA boundaries
# lga = gpd.read_file(LGA_SHAPEFILE)
# lga.plot(edgecolor='black', linewidth=0.2, figsize=(8, 8))
# plt.title('Nigeria LGA Boundaries')
# plt.axis('off')

## 4. Scratch

In [ ]:
# Free scratch space — add cells as needed